# 02 - Cohort Construction

keep members continuously enrolled in part a + b, no hmo/medicare advantage, in **every** year (2008, 2009, 2010), alive through the label window.

In [ ]:
import duckdb
import yaml
from pathlib import Path

with open("../configs/data_sources.yaml") as f:
    config = yaml.safe_load(f)

interim_dir = Path("..") / config["interim_dir"]
processed_dir = Path("..") / config["processed_dir"]
processed_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# register each year's beneficiary summary as its own view -- one
# beneficiary has one row PER YEAR, so "continuously enrolled" has to be
# checked against all three years, not one combined table
con = duckdb.connect()
for year in (2008, 2009, 2010):
    con.execute(f"""
        create view beneficiary_summary_{year} as
        select * from read_parquet('{interim_dir}/beneficiary_summary_{year}.parquet')
    """)


In [ ]:
# load the cohort sql and bind the label-window end date
# dates in this data are plain YYYYMMDD integers (e.g. 20101231), not a real
# date type, so the bound value is an int literal, not a quoted string
sql = Path("../sql/cohort/build_cohort.sql").read_text()
label_window_end = int(config["study_period"]["end"].replace("-", ""))
sql = sql.replace(":label_window_end", str(label_window_end))

cohort = con.execute(sql).df()
print(f"cohort size: {len(cohort):,} members")
cohort.head()


In [ ]:
# persist the cohort for downstream feature engineering
out_path = processed_dir / "cohort.parquet"
cohort.to_parquet(out_path, index=False)
print(f"wrote {out_path}")
